# core

> Claude code api backend for fastllm

FastLLM's `claude_code` provider is a thin adapter over `fastclaude`. `claude_mk_payload` hands the initial canonical history and tool schemas to `ClaudeRun`; when Claude calls a tool, FastLLM executes it and continues the same paused run using `previous_response_id`. Only a paused tool response carries a response id: it is a single-use handle to the Claude process and pending MCP calls inside the adapter. Terminal responses carry no handle, so later user turns replay canonical history through a fresh process. Retained handles expire automatically, while explicit cancellation reaches `ClaudeRun.aclose`, interrupts natively, and reaps the process.

In [ ]:
#| default_exp core

In [ ]:
#| export
import asyncio, uuid
from fastcore.utils import *
from fastllm.types import *
from aidialog.msg_parts import Completion, ToolResult
from fastllm.anthropic import norm_sse_event, norm_tool_calls, norm_parts, norm_finish, norm_usage, finalize_usage, delta_index_fn, cost
from fastllm.streaming import mk_acollect_stream, Delta
from fasttransport.errors import APIError
from fastclaude.core import astream, unqual, SERVER_TOOLS

In [ ]:
from fastllm.chat import AsyncChat, lite_mk_func, mk_msgs, mk_tool_res_msg
from aidialog.msg_parts import ToolUse
from fastcore.test import *

`claude_mk_payload` distinguishes the two standard Responses-style requests. An initial request passes the canonical history to `astream`; a continuation carries only its `previous_response_id` and the new `ToolResult` parts, because the paused `ClaudeRun` already owns the earlier conversation. The payload also carries the retention TTL for a paused process. Both Responses and Chat Completions function schemas map to fastclaude's, and `web_search_options` enables Claude Code's own search tools:

In [ ]:
#| export
def claude_mk_payload(msgs, model, stream=False, **kwargs):
    "Build initial `astream` inputs or a response-id continuation"
    tools = [dict(name=f['name'], description=f.get('description',''), inputSchema=f.get('parameters', {}))
        for t in (kwargs.get('tools') or []) if t.get('type') == 'function' and (f := t.get('function') or t)]
    native = SERVER_TOOLS if kwargs.get('web_search_options') is not None else ()
    previous = kwargs.get('previous_response_id')
    payload = dict(model=model, previous_response_id=previous, response_ttl=kwargs.get('response_ttl', 3600))
    if previous: payload['results'] = [p for m in msgs for p in m.content if isinstance(p, ToolResult)]
    else: payload.update(msgs=list(msgs), system=kwargs.get('system') or '', tools=tools or None, native_tools=native)
    return payload

In [ ]:
def simple_add(a: int, b: int) -> int:
    "Add two numbers"
    return a + b

p = claude_mk_payload(mk_msgs(['What is 2+2?']), 'claude-sonnet-5', tools=[lite_mk_func(simple_add)])
test_eq(p['native_tools'], ())
rp = claude_mk_payload([], 'm', tools=[dict(type='function', name='py', parameters={'type':'object'})])
test_eq(rp['tools'][0]['name'], 'py')
p['tools'][0]

In [ ]:
test_eq(claude_mk_payload([], 'm', web_search_options='l')['native_tools'], SERVER_TOOLS)
tmsg = mk_tool_res_msg([ToolUse(id='call_1', name='simple_add', arguments={})], ['4'])
cp = claude_mk_payload([tmsg], 'claude-sonnet-5', previous_response_id='resp_1', response_ttl=15)
test_eq((cp['previous_response_id'], cp['results'], cp['response_ttl']), ('resp_1', tmsg.content, 15))

The stream adapter re-indexes partial events onto one global block sequence (`_reidx`), because thinking, text, and the tool call arrive as separate wire messages that each restart at index 0, then normalizes them to FastLLM `Delta`s with tool names unqualified.

In [ ]:
#| export
def _reidx():
    "Stateful rebase of per-message block indices onto one global sequence"
    base,mx = 0,-1
    def f(ev):
        nonlocal base,mx
        t = ev.get('type')
        if t=='message_start': base,mx = base+mx+1,-1
        elif t in ('content_block_start','content_block_delta','content_block_stop') and 'index' in ev:
            mx = max(mx, ev['index'])
            ev = {**ev, 'index': ev['index']+base}
        return ev
    return f


In [ ]:
reindex = _reidx()
indices = [reindex(dict(type='content_block_start', index=0))['index'],
    reindex(dict(type='content_block_start', index=1))['index']]
reindex(dict(type='message_start'))
indices.append(reindex(dict(type='content_block_start', index=0))['index'])
test_eq(indices, [0,1,2])
indices

A tool-call response stores the paused `ClaudeRun` under its new response id; the continuation removes that entry, supplies the complete tool-result batch, and resumes the same process.

In [ ]:
#| export
_claude_runs,_claude_timers = {},{}

`_take_run` is the one-shot operation behind continuation: removing the run and cancelling its expiry timer happen together, so the same response id cannot be resumed twice.

In [ ]:
#| export
def _take_run(response_id):
    run = _claude_runs.pop(response_id, None)
    if timer := _claude_timers.pop(response_id, None): timer.cancel()
    return run

In [ ]:
held_run = object()
_claude_runs['resp_demo'] = held_run
test_is(_take_run('resp_demo'), held_run)
test_eq(_take_run('resp_demo'), None)
_claude_runs

Terminal responses close without an id, a timer expires abandoned pauses, and `claude_cancel` releases one immediately:

In [ ]:
#| export
async def _expire_run(response_id, ttl):
    await asyncio.sleep(ttl)
    run = _claude_runs.pop(response_id, None)
    _claude_timers.pop(response_id, None)
    if run is not None: await run.aclose()

Only the paused completion receives that id, marked non-reusable because resuming consumes the process.

In [ ]:
#| export
def _keep_run(response_id, run, ttl):
    _claude_runs[response_id] = run
    _claude_timers[response_id] = asyncio.create_task(_expire_run(response_id, ttl))

`claude_cancel` uses that same consuming lookup, then closes the process immediately. Returning a boolean lets an HTTP or client adapter distinguish a real cancellation from an already consumed or expired id.

In [ ]:
#| export
async def claude_cancel(response_id):
    "Close the paused run identified by `response_id`; return whether it existed"
    run = _take_run(response_id)
    if run is None: return False
    await run.aclose()
    return True

Claude CLI emits one empty `partial_json` chunk for a zero-argument tool. FastLLM's collector normally recognizes a tool only when arguments finish as JSON, so `_ToolBlocks` remembers every opened tool block and which ones actually completed JSON.

In [ ]:
#| export
class _ToolBlocks:
    def __init__(self): self.open,self.complete = {},set()

`observe` updates that state after each normalized event. When an unfinished block closes, it returns one synthetic empty-arguments delta; ordinary and completed blocks return nothing extra.

In [ ]:
#| export
@patch
def observe(self:_ToolBlocks, ev, delta):
    et,idx = ev.get('type'),ev.get('index')
    if et=='content_block_start' and delta.tool_calls: self.open[idx] = delta.tool_calls[0]
    elif et=='content_block_delta' and delta.tool_calls and (nested_idx(ev, 'delta', 'partial_json') or '').endswith('}'):
        self.complete.add(idx)
    if et!='content_block_stop' or (call := self.open.pop(idx, None)) is None or idx in self.complete: return
    return Delta(tool_calls=[ToolUse(id=call.id, name=call.name, arguments={})], raw=dict(index=idx))

In [ ]:
blocks = _ToolBlocks()
empty_call = ToolUse(id='call_0', name='ping', arguments={})
test_eq(blocks.observe(dict(type='content_block_start', index=0), Delta(tool_calls=[empty_call])), None)
empty_delta = blocks.observe(dict(type='content_block_stop', index=0), Delta())
test_eq(empty_delta.tool_calls[0].arguments, {})
empty_delta

Each response is one generator pipeline: raw run events are filtered to partials, re-indexed, normalized, and collected by FastLLM's standard collector.

The calls are not marked server-executed: they are the chat's own tools, and FastLLM's loop must run them.

In [ ]:
#| export
async def _claude_deltas(run):
    "Normalize one ClaudeRun turn to FastLLM deltas"
    reindex = _reidx()
    blocks = _ToolBlocks()
    async for m in run:
        if m.get('type')!='stream_event': continue
        ev = reindex(m['event'])
        d = norm_sse_event(ev)
        for tc in (d.tool_calls or []): tc.name = unqual(tc.name)
        yield d
        if extra := blocks.observe(ev, d): yield extra

Starting a logical turn has two mutually exclusive paths. A fresh request creates a new `ClaudeRun`; a continuation consumes its response id, rejects a missing handle as a conflict, supplies the complete result batch, and returns the resumed run.

In [ ]:
#| export
async def _claude_run(payload):
    previous = payload.get('previous_response_id')
    if not previous: return astream(**{k:v for k,v in payload.items() if k not in ('previous_response_id','response_ttl')})
    run = _take_run(previous)
    if run is None:
        raise APIError(f'unknown, expired, or consumed Claude response: {previous}', provider='claude_code',
            model=payload.get('model'), status_code=409, code='invalid_previous_response_id')
    await run.resume(payload.get('results') or [])
    return run

Only a final `Completion` from a paused run becomes a continuation. `_retain_run` annotates that completion as one-shot, starts its expiry timer, and reports whether the collector must leave the process open.

In [ ]:
#| export
def _retain_run(completion, run, response_id, ttl):
    if not isinstance(completion, Completion) or not run.paused: return False
    completion.raw.update(response_id=response_id, response_id_reusable=False)
    _keep_run(response_id, run, ttl)
    return True

Claude CLI reports provider failures in its terminal result rather than raising them through iteration. `_check_run` translates that terminal shape to FastLLM's ordinary provider error after all streamed parts have been collected.

In [ ]:
#| export
def _check_run(run, payload):
    if not run.result or not run.result.get('is_error'): return
    raise APIError(str(run.result.get('result') or run.result.get('subtype')), provider='claude_code',
        model=payload.get('model'), status_code=run.result.get('api_error_status'), raw=run.result)

`claude_acollect_stream` is now only the lifecycle composition: obtain the logical run, normalize and collect its deltas, retain it only when the completion is paused, translate a terminal provider error, and otherwise close it in `finally`.

In [ ]:
#| export
async def claude_acollect_stream(payload, **kwargs):
    "Adapt one logical `ClaudeRun` turn to FastLLM, retaining paused runs by response id"
    run,keep = None,False
    try:
        run = await _claude_run(payload)
        rid = f'resp_{uuid.uuid4().hex}'

        async for o in mk_acollect_stream(_claude_deltas(run), index_fn=delta_index_fn, api_name='claude_code', **kwargs):

            if _retain_run(o, run, rid, payload['response_ttl']): keep = True
            yield o

        _check_run(run, payload)
    finally:
        if run is not None and not keep: await run.aclose()

The provider registration supplies FastLLM's standard Anthropic normalization and costing functions, while declaring that this transport can continue through a one-shot response id.

In [ ]:
#| export
api_registry.register('claude_code',
    norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    supports_previous_response_id=True, finalize_usage=finalize_usage, mk_payload=claude_mk_payload, acollect_stream=claude_acollect_stream, cost=cost)

## Live runs

Against the real CLI (genuine captured output; spends tokens, so excluded from automated runs), one FastLLM chat drives the client-owned tool loop. The first response pauses at `simple_add`; FastLLM executes it and continues through the returned response id. The same Claude process produces the final answer, then the adapter clears its continuation handle so a later top-level turn can replay the familiar canonical history:

In [ ]:
#| eval: false
chat = AsyncChat('claude_code/claude-sonnet-5', tools=[simple_add], use_previous_response_id=True)
rs = await chat('What is 7+3? Use the tool, then answer with only the number.', stream=True, max_steps=3)
parts = [o async for o in rs]
test_eq([m.role for m in chat.hist], ['user','assistant','tool','assistant'])
test_eq(chat.hist[-1].text, '10')
test('"result": "10"', chat.full(), in_)
test_eq((_claude_runs, _claude_timers), ({}, {}))
assert chat.response_id is None
[type(o).__name__ for o in parts]

['ToolUse', 'Completion', 'ToolResult', 'Refresh', 'Text', 'Completion']